# Visualize ANN Benchmark Results

Presentation-focused view of the results produced by `benchmark_ann.py`
(and `pipeline/corruption.py` for robustness). The analysis notebook is
`compare_methods.ipynb`; this one only draws clean figures.

| Config | Model | Dataset |
|---|---|---|
| `vit-coco` | ViT-B/16 | COCO unlabeled2017 |
| `resnet-coco` | ResNet18 | COCO unlabeled2017 |
| `vit-zappos` | ViT-B/16 | UT-Zap50K |
| `resnet-zappos` | ResNet18 | UT-Zap50K |

Every figure is also saved as a PNG to `media/` at the repository root.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Configurations to include (keys from datasets.py).
CONFIG_KEYS = ["vit-coco", "resnet-coco", "vit-zappos", "resnet-zappos"]

SAVE_FIGURES = True   # write every figure to media/
FIG_DPI = 200

In [ ]:
# Find this experiment folder whether the notebook runs from its own
# folder or from the repository root, then reuse the shared registry.
EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "datasets.py").exists():
    EXPERIMENT_DIR = Path.cwd() / "experiments" / "ann-comparison"
sys.path.insert(0, str(EXPERIMENT_DIR))
from datasets import CONFIGS, DATASETS, MODELS, REPO_ROOT, results_dir

MEDIA_DIR = REPO_ROOT / "media"
MEDIA_DIR.mkdir(exist_ok=True)

# Search parameter swept by each method (Exact FlatIP has none).
SWEEP_PARAM = {"LSH": "nbits", "IVF-Flat": "nprobe", "IVF-PQ": "nprobe",
               "HNSW": "efSearch", "HNSW-PQ": "efSearch"}

meta_rows, result_rows = [], []
for key in CONFIG_KEYS:
    path = results_dir(key) / "ann_results.json"
    if not path.exists():
        print(f"{key}: no ann_results.json - run benchmark_ann.py --config {key}")
        continue
    data = json.loads(path.read_text())
    meta_rows.append({"config": key, **data["metadata"]})
    for r in data["results"]:
        param = SWEEP_PARAM.get(r["algorithm"])
        result_rows.append({
            "config": key,
            "label": CONFIGS[key]["label"],
            "model": CONFIGS[key]["model"],
            "dataset": CONFIGS[key]["dataset"],
            "algorithm": r["algorithm"],
            "param_name": param,
            "param_value": r["parameters"][param] if param else np.nan,
            "setting": f"{param}={r['parameters'][param]}" if param else "-",
            **{k: v for k, v in r.items() if k not in ("algorithm", "parameters")},
        })

# meta: one row per configuration. results: one row per (config, method, setting).
meta = pd.DataFrame(meta_rows).set_index("config")
results = pd.DataFrame(result_rows)
results["model_label"] = results["model"].map(lambda m: MODELS[m]["label"])
results["dataset_label"] = results["dataset"].map(lambda d: DATASETS[d]["label"])

# Relative to exact search in the same configuration (raw latencies are not
# comparable across configs: gallery size and dimension differ).
exact = results[results["algorithm"] == "Exact FlatIP"].set_index("config")
results["speedup"] = results["config"].map(exact["latency_ms_per_query"]) / results["latency_ms_per_query"]
results["compression"] = results["config"].map(exact["index_size_mb"]) / results["index_size_mb"]

# Corruption robustness summaries (optional, from pipeline/corruption.py).
corruption_frames = []
for key in meta.index:
    path = results_dir(key) / "robustness" / "corruption_summary.csv"
    if path.exists():
        corruption_frames.append(pd.read_csv(path).assign(
            config=key, model=CONFIGS[key]["model"], dataset=CONFIGS[key]["dataset"]))
corruption = pd.concat(corruption_frames, ignore_index=True) if corruption_frames else None

display(meta[["label", "gallery_size", "num_queries", "dimension"]])
print(f"results: {len(results)} rows x {results.shape[1]} columns")
print(f"corruption results: {'none' if corruption is None else ', '.join(corruption['config'].unique())}")
print(f"figures -> {MEDIA_DIR}")

In [ ]:
results.head()

In [ ]:
# Shared helpers and styling, reused by every figure below.
METHOD_COLORS = {
    "Exact FlatIP": "tab:gray",
    "LSH": "tab:green",
    "IVF-Flat": "tab:red",
    "IVF-PQ": "tab:orange",
    "HNSW": "tab:blue",
    "HNSW-PQ": "tab:purple",
}
METHOD_ORDER = list(METHOD_COLORS)


def save_figure(fig, name):
    # Save to media/<name>.png when SAVE_FIGURES is on.
    if not SAVE_FIGURES:
        return
    path = MEDIA_DIR / f"{name}.png"
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    print(f"saved {path.relative_to(REPO_ROOT)}")


def pick(algorithm, param_value=None):
    # Rows of one method (optionally one setting) across all configurations.
    sub = results[results["algorithm"] == algorithm]
    if param_value is not None:
        sub = sub[sub["param_value"] == param_value]
    return sub


# One colour per experiment (model x dataset), shared by every per-config chart.
CONFIG_COLORS = {
    "vit-coco": "tab:blue",
    "resnet-coco": "tab:orange",
    "vit-zappos": "tab:green",
    "resnet-zappos": "tab:red",
}

# (column, display label) for every retrieval accuracy metric.
ACCURACY_METRICS = [
    (f"{metric}_at_{k}", f"{label}@{k}")
    for metric, label in [("recall", "Recall"), ("map", "mAP")]
    for k in (5, 10, 20)
]


def plot_sweep(ax, algorithm, metric, ylabel, ylim=(0, 1)):
    # One line per configuration: `metric` against the method's swept parameter.
    sub = pick(algorithm)
    for config in meta.index:
        rows = sub[sub["config"] == config].sort_values("param_value")
        ax.plot(rows["param_value"], rows[metric], marker="o",
                color=CONFIG_COLORS[config], label=meta.loc[config, "label"])
    values = sorted(sub["param_value"].unique())
    ax.set_xscale("log", base=2)
    ax.set_xticks(values, [f"{v:g}" for v in values])
    ax.minorticks_off()
    ax.set_xlabel(sub["param_name"].iloc[0])
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.grid(alpha=0.3)


def accuracy_figures(algorithm, file_prefix, ylim=(0, 1)):
    # One figure per accuracy metric -> media/<file_prefix>_<metric>.png, then
    # all metrics in one grid (rows Recall / mAP, columns @5 / @10 / @20)
    # -> media/<file_prefix>_accuracy_grid.png. `ylim` zooms the score axis
    # when every value sits in a narrow band.
    param = pick(algorithm)["param_name"].iloc[0]
    for metric, label in ACCURACY_METRICS:
        fig, ax = plt.subplots(figsize=(6, 4))
        plot_sweep(ax, algorithm, metric, label, ylim)
        ax.set_title(f"{algorithm}: {label} vs {param}")
        ax.legend(fontsize=8)
        save_figure(fig, f"{file_prefix}_{metric}")
        plt.show()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
    for ax, (metric, label) in zip(axes.ravel(), ACCURACY_METRICS):
        plot_sweep(ax, algorithm, metric, label, ylim)
        ax.set_title(label)
        ax.label_outer()   # axis labels only on the outer panels
    for ax, row_label in zip(axes[:, 0], ["Recall", "mAP"]):
        ax.set_ylabel(row_label)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
    fig.suptitle(f"{algorithm} retrieval accuracy vs {param}", fontsize=14)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    save_figure(fig, f"{file_prefix}_accuracy_grid")
    plt.show()

## Visualizations

Each figure gets its own section and is saved to `media/` with
`save_figure(fig, "<name>")`.

### LSH: retrieval accuracy vs nbits

`nbits` is the length of the binary hash code each vector is reduced to.
More bits split the space into finer buckets, so the hashed neighbours are
closer to the true ones, but the index grows and search gets slower.

Every metric is measured against the **exact top-K of the same
configuration** (one line per model x dataset experiment):

- **Recall@K**: fraction of the exact top-K that LSH also returns.
- **mAP@K**: rewards placing the true neighbours near the top of the list,
  so wrong ordering lowers it.

Precision@K is not shown: it always equals Recall@K here, because both
lists hold exactly K items, so each is *hits / K*.

In [ ]:
accuracy_figures("LSH", "lsh")

### IVF-Flat: retrieval accuracy vs nprobe

IVF splits the gallery into `nlist = 256` clusters (k-means on the
embeddings). A query is compared only against the vectors in its
`nprobe` nearest clusters, so a larger `nprobe` searches more of the
gallery: higher accuracy, slower search. At `nprobe = 256` it would be
exact search. IVF-Flat stores the full, uncompressed vectors, so any miss
comes from a true neighbour sitting in a cluster that was not probed.

In [ ]:
accuracy_figures("IVF-Flat", "ivfflat")

### IVF-PQ: retrieval accuracy vs nprobe

Same clustering and `nprobe` as IVF-Flat, but each vector is compressed
with product quantization: split into `m` sub-vectors (48 for ViT, 32 for
ResNet18), each stored as one 8-bit code. Distances are approximate, so
accuracy levels off below IVF-Flat even when many clusters are probed; the
gap between the two sets of charts is the cost of compression.

In [ ]:
accuracy_figures("IVF-PQ", "ivfpq")

### HNSW: retrieval accuracy vs efSearch

HNSW links the gallery vectors into a layered nearest-neighbour graph
(`M = 32` links per node, built with `efConstruction = 200`) and answers a
query by walking that graph towards it. `efSearch` is the size of the
candidate list kept during the walk: a larger list explores more of the
graph, so accuracy rises and search gets slower. Vectors are stored
uncompressed, so distances are exact.

Every value is above 0.85, so the score axis is **zoomed to 0.84-1.0**
(the other charts use 0-1) to make the differences between experiments
visible.

In [ ]:
accuracy_figures("HNSW", "hnsw", ylim=(0.84, 1.0))

### HNSW-PQ: retrieval accuracy vs efSearch

Same graph and `efSearch` as HNSW, but the graph walk uses product-quantized
vectors (`m` = 48 sub-vectors for ViT, 32 for ResNet18, 8 bits each).
The walk is guided by approximate distances, so it can step towards the
wrong nodes, and accuracy stays well below plain HNSW.

In [ ]:
accuracy_figures("HNSW-PQ", "hnswpq")